In [39]:
import pandas as pd
import numpy as np
import time

import sys
import os

import requests
import json
from pathlib import Path, PurePath
from datetime import datetime

In [40]:
# Token for API access
token = "338f417f5f8ae25ba6eb01c878134153FE905CC0A125F5E827810F937A192125F15C8769"

In [41]:
def wialon_login(token, full=False):
    url = "https://hst-api.wialon.com/wialon/ajax.html"

    params = {
        "svc": "token/login",
        "params": json.dumps({"token": token})
    }

    response = requests.post(url, params=params)
    result = response.json()

    if full:
        return result
    return result.get("eid")

In [42]:
eid = wialon_login(token)
print("EID:", eid)

EID: 10174ee2655333aa03b26af26b0a914c


In [43]:
import pandas as pd
from datetime import datetime, timezone

# Explicit UTC interval: 22 Feb 2026 07:00 → 23 Feb 2026 15:30
start_utc = datetime(2026, 4, 26, 0, 0, tzinfo=timezone.utc)
end_utc = datetime(2026, 4, 27, 23, 59, tzinfo=timezone.utc)

from_ts = int(start_utc.timestamp())
to_ts = int(end_utc.timestamp())

# 1) Execute speeding report for West Kenya (same parameters as Wialon UI)
exec_payload = {
    "svc": "report/exec_report",
    "params": json.dumps(
        {
            "reportResourceId": 17082202,
            "reportTemplateId": 220,
            "reportObjectId": 30182477,
            "reportObjectSecId": 0,
            "interval": {
                "flags": 0,
                "from": from_ts,
                "to": to_ts,
            },
        }
    ),
    "sid": eid,
}

exec_response = requests.post(
    "https://hst-api.wialon.com/wialon/ajax.html",
    data=exec_payload,
)
exec_result = exec_response.json()

# 2) Helper function to fetch any table by index
def fetch_table(report_tables, table_index):
    if table_index >= len(report_tables):
        print(f"Table index {table_index} not found. Only {len(report_tables)} table(s) available.")
        return pd.DataFrame()

    table_meta = report_tables[table_index]
    headers = table_meta.get("header", [])
    row_count = table_meta.get("rows", 0)

    rows_payload = {
        "svc": "report/get_result_rows",
        "params": json.dumps(
            {
                "tableIndex": table_index,   # <-- key change: use the target index
                "indexFrom": 0,
                "indexTo": max(row_count - 1, 0),
            }
        ),
        "sid": eid,
    }

    rows_response = requests.post(
        "https://hst-api.wialon.com/wialon/ajax.html",
        data=rows_payload,
    )
    rows_json = rows_response.json()

    rows_data = []
    for row in rows_json:
        cells = row.get("c", [])
        values = [c.get("t") if isinstance(c, dict) else c for c in cells]
        rows_data.append(values)

    if rows_data:
        max_cols = min(len(headers), len(rows_data[0]))
        return pd.DataFrame(rows_data, columns=headers[:max_cols])
    else:
        return pd.DataFrame(columns=headers)


report_tables = exec_result.get("reportResult", {}).get("tables", [])

# Fetch both sheets
ena_coach_summary  = fetch_table(report_tables, table_index=0)  # Sheet 1
ena_coach_ecodriving = fetch_table(report_tables, table_index=1)  # Sheet 2
ena_coach_location = fetch_table(report_tables, table_index=2)  # Sheet 3
ena_coach_fillings = fetch_table(report_tables, table_index=3)  # Sheet 4
ena_coach_drains = fetch_table(report_tables, table_index=4)  # Sheet 5
ena_coach_idling = fetch_table(report_tables, table_index=5)  # Sheet 6
ena_coach_Speed = fetch_table(report_tables, table_index=6)  # Sheet 7



In [44]:
ena_coach_Speed

,Grouping,Driver,Initial location,Final location,Mileage,Avg. speed,Duration
0,ENA COACH - KDE 181Q,,,,0.00 km,0 km/h,0:00:00
1,ENA COACH - KDE 182Q,,"Birongo-Kisii Road, Kenya, Keroka","Mombasa Road, Kenya, 1.31 km from Syokimau",573 km,45 km/h,12:41:31


In [45]:
ena_coach_idling

,Grouping,Initial location,Final location,Engine hours,In motion,Idling,Driver
0,ENA COACH - KDE 181Q,"Mombasa Road, Kenya, 1.28 km from Syokimau","Mombasa Road, Kenya, 1.31 km from Syokimau",1:33:24,0:01:26,1:31:58,ENA Driver1
1,ENA COACH - KDE 182Q,"Birongo-Kisii Road, Kenya, Keroka","Mombasa Road, Kenya, 1.30 km from Syokimau",15:59:53,11:52:53,4:07:00,


In [46]:
ena_coach_fillings

,Grouping,Filling or charge time,Location,Initial fuel level,Filled,Final fuel level,Registered filling,Filling difference,Driver,Mileage
0,ENA COACH - KDE 181Q,27.04.2026 05:48:23,"Mombasa Road, Kenya, 1.28 km from Syokimau",351 l,27 l,378 l,-----,27 l,ENA Driver1,1251202 km
1,ENA COACH - KDE 182Q,-----,,-----,-----,-----,-----,0.00 l,,1276087 km


In [47]:
ena_coach_drains

,Grouping,Initial location,Drain time,Final location,Drained,Final fuel level,Driver,Mileage,Initial fuel level
0,ENA COACH - KDE 181Q,,-----,,-----,-----,,1251202 km,-----
1,ENA COACH - KDE 182Q,,-----,,-----,-----,,1276087 km,-----


In [48]:
ena_coach_location

,Grouping,Last message time,Last coordinates time,Location,Driver
0,ENA COACH - KDE 181Q,27.04.2026 12:17:38,27.04.2026 12:17:38,"Mombasa Road, Kenya, 1.31 km from Syokimau",
1,ENA COACH - KDE 182Q,27.04.2026 12:19:04,27.04.2026 12:19:04,"Mombasa Road, Kenya, 1.30 km from Syokimau",


In [49]:
ena_coach_summary

,Grouping,Mileage in all messages,Avg. speed,Engine hours,Avg. mileage per unit of fuel by FLS,Total fillings,Total drains,Filled,Drained,Consumed by AbsFCS,Avg. consumption by AbsFCS,Max. value of custom sensor,Max. speed
0,ENA COACH - KDE 181Q,0.15 km,0 km/h,1:33:24,0.00 km,1,0,27 l,0.00 l,1.00 l,667 l/100 km,-----,6 km/h
1,ENA COACH - KDE 182Q,574 km,16 km/h,15:59:53,0.00 km,0,0,0.00 l,0.00 l,194 l,34 l/100 km,-----,108 km/h


In [50]:
def detailize_table(table_index, row_index, col_index=0):
    """
    Fetch detailization (subrows) for a specific row in a report table
    """
    detail_payload = {
        "svc": "report/get_result_subrows",
        "params": json.dumps(
            {
                "tableIndex": table_index,
                "rowIndex": row_index,
                "colIndex": col_index,  # usually 0 works, but depends on report structure
                "indexFrom": 0,
                "indexTo": 1000,  # adjust if you expect more rows
            }
        ),
        "sid": eid,
    }

    response = requests.post(
        "https://hst-api.wialon.com/wialon/ajax.html",
        data=detail_payload,
    )

    result = response.json()

    rows_data = []
    for row in result:
        cells = row.get("c", [])
        values = [c.get("t") if isinstance(c, dict) else c for c in cells]
        rows_data.append(values)

    return rows_data

In [51]:
all_details = []

table_index = 1  # ecodriving table

for i in range(len(ena_coach_ecodriving)):
    details = detailize_table(table_index=table_index, row_index=i)

    for d in details:
        all_details.append([i] + d)  # attach parent row index

# Convert to DataFrame
detail_df = pd.DataFrame(all_details)

num_cols = detail_df.shape[1]

detail_columns = ["parent_row"] + [f"field_{i}" for i in range(1, num_cols)]

detail_df.columns = detail_columns

In [52]:
# Drop unwanted columns
detail_df = detail_df.drop(columns=["parent_row", "field_1", "field_12"], errors="ignore")

# Keep only expected number of columns
detail_df = detail_df.iloc[:, :100000]

# Rename columns
detail_df.columns = [
    "Grouping",
    "Violation",
    "Beginning",
    "Initial location",
    "End",
    "Final location",
    "Avg. speed",
    "Max. speed",
    "Duration",
    "Mileage",
    "Count"
]

In [53]:
detail_df.head()

,Grouping,Violation,Beginning,Initial location,End,Final location,Avg. speed,Max. speed,Duration,Mileage,Count
0,ENA COACH - KDE 181Q,Harsh Braking,27.04.2026 06:18:15,"Mombasa Road, Kenya, 1.29 km from Syokimau",27.04.2026 06:18:27,"Mombasa Road, Kenya, 1.29 km from Syokimau",1.00,0 km/h,0:00:12,0.00 km,1
1,ENA COACH - KDE 181Q,Accelerator &gt; 70%,27.04.2026 06:19:53,"Mombasa Road, Kenya, 1.29 km from Syokimau",27.04.2026 06:20:15,"Mombasa Road, Kenya, 1.28 km from Syokimau",101.00,0 km/h,0:00:22,0.00 km,1
2,ENA COACH - KDE 181Q,Accelerator &lt; 40 %,27.04.2026 06:19:53,"Mombasa Road, Kenya, 1.29 km from Syokimau",27.04.2026 06:20:15,"Mombasa Road, Kenya, 1.28 km from Syokimau",101.00,0 km/h,0:00:22,0.00 km,1
3,ENA COACH - KDE 181Q,Engine Temp &gt;105°,27.04.2026 06:19:55,"Mombasa Road, Kenya, 1.28 km from Syokimau",27.04.2026 06:20:15,"Mombasa Road, Kenya, 1.28 km from Syokimau",214.00,0 km/h,0:00:20,0.00 km,1
4,ENA COACH - KDE 181Q,Harsh Braking,27.04.2026 06:20:15,"Mombasa Road, Kenya, 1.28 km from Syokimau",27.04.2026 06:20:25,"Mombasa Road, Kenya, 1.28 km from Syokimau",1.00,0 km/h,0:00:10,0.00 km,1


In [54]:
detail_df['Violation'].count()

np.int64(2349)

In [55]:
unique_violations = detail_df['Violation'].unique().tolist()
unique_violations

['Harsh Braking',
 'Accelerator &gt; 70%',
 'Accelerator &lt; 40 %',
 'Engine Temp &gt;105°',
 'Green Band Driving',
 'Engine Stress',
 'Over Speeding',
 'Harsh Acceleration',
 'Over Revving']

In [56]:
from urllib.parse import quote_plus
from IPython.display import display, HTML


def make_location_hyperlink(location: str) -> str:
    """Return clickable Google Maps search link for a location string."""
    if pd.isna(location) or str(location).strip() == "":
        return ""
    loc = str(location).strip()
    url = f"https://www.google.com/maps/search/?api=1&query={quote_plus(loc)}"
    return f'<a href="{url}" target="_blank">{loc}</a>'


violation_order = [
    "Harsh Cornering",
    "Over Speeding",
    "Harsh Braking",
    "Free Wheeling",
    "Over Revving",
]

violation_tables = {}

for violation in violation_order:
    vdf = detail_df[detail_df["Violation"] == violation].copy()

    # Convert location columns to hyperlinks
    for col in ["Initial location", "Final location"]:
        if col in vdf.columns:
            vdf[col] = vdf[col].apply(make_location_hyperlink)

    violation_tables[violation] = vdf

In [57]:
harsh_cornering_df = detail_df[detail_df["Violation"] == "Harsh Cornering"].copy()
for col in ["Initial location", "Final location"]:
    harsh_cornering_df[col] = harsh_cornering_df[col].apply(make_location_hyperlink)
harsh_cornering_df.index = range(1, len(harsh_cornering_df) + 1)
HTML(harsh_cornering_df.head(5).to_html(escape=False))

,Grouping,Violation,Beginning,Initial location,End,Final location,Avg. speed,Max. speed,Duration,Mileage,Count


In [58]:
over_speeding_df = detail_df[detail_df["Violation"] == "Over Speeding"].copy()
for col in ["Initial location", "Final location"]:
    over_speeding_df[col] = over_speeding_df[col].apply(make_location_hyperlink)
over_speeding_df.index = range(1, len(over_speeding_df) + 1)
HTML(over_speeding_df.head(5).to_html(escape=False))

,Grouping,Violation,Beginning,Initial location,End,Final location,Avg. speed,Max. speed,Duration,Mileage,Count
1,ENA COACH - KDE 182Q,Over Speeding,26.04.2026 00:06:42,"Birongo-Kisii Road, Kenya, 2.33 km from Amabuko",26.04.2026 00:06:46,"Birongo-Kisii Road, Kenya, 2.42 km from Amabuko",1.00,82 km/h,0:00:04,0.09 km,1
2,ENA COACH - KDE 182Q,Over Speeding,26.04.2026 01:00:55,"Rongo-Kisii Road, Kenya, 3.13 km from Suneka",26.04.2026 01:00:57,"Rongo-Kisii Road, Kenya, 3.16 km from Suneka",1.00,83 km/h,0:00:02,0.03 km,1
3,ENA COACH - KDE 182Q,Over Speeding,26.04.2026 01:01:15,"Rongo-Kisii Road, Kenya, 3.48 km from Suneka",26.04.2026 01:01:19,"Rongo-Kisii Road, Kenya, 3.57 km from Suneka",1.00,81 km/h,0:00:04,0.07 km,1
4,ENA COACH - KDE 182Q,Over Speeding,26.04.2026 01:04:42,"Rongo-Kisii Road, Kenya, 4.20 km from Tabaka",26.04.2026 01:04:43,"Rongo-Kisii Road, Kenya, 4.16 km from Tabaka",1.00,85 km/h,0:00:01,0.03 km,1
5,ENA COACH - KDE 182Q,Over Speeding,26.04.2026 01:06:14,"Rongo-Kisii Road, Kenya, 3.91 km from Tabaka",26.04.2026 01:06:17,"Rongo-Kisii Road, Kenya, 3.91 km from Tabaka",1.00,86 km/h,0:00:03,0.07 km,1


In [59]:
free_wheeling_df = detail_df[detail_df["Violation"] == "Free Wheeling"].copy()
for col in ["Initial location", "Final location"]:
    free_wheeling_df[col] = free_wheeling_df[col].apply(make_location_hyperlink)
free_wheeling_df.index = range(1, len(free_wheeling_df) + 1)
HTML(free_wheeling_df.head(5).to_html(escape=False))

,Grouping,Violation,Beginning,Initial location,End,Final location,Avg. speed,Max. speed,Duration,Mileage,Count


In [60]:
over_revving_df = detail_df[detail_df["Violation"] == "Over Revving"].copy()
for col in ["Initial location", "Final location"]:
    over_revving_df[col] = over_revving_df[col].apply(make_location_hyperlink)
over_revving_df.index = range(1, len(over_revving_df) + 1)
HTML(over_revving_df.head(5).to_html(escape=False))

,Grouping,Violation,Beginning,Initial location,End,Final location,Avg. speed,Max. speed,Duration,Mileage,Count
1,ENA COACH - KDE 182Q,Over Revving,26.04.2026 01:35:40,"A1, Kenya, Elisha",26.04.2026 01:35:42,"A1, Kenya, Elisha",1834.00,46 km/h,0:00:02,0.02 km,1
2,ENA COACH - KDE 182Q,Over Revving,26.04.2026 02:37:00,"A1, Kenya, 2.19 km from Migori",26.04.2026 02:37:04,"A1, Kenya, 2.24 km from Migori",1812.00,47 km/h,0:00:04,0.07 km,1
3,ENA COACH - KDE 182Q,Over Revving,26.04.2026 09:14:57,"B6, Kenya, 2.88 km from Sotik",26.04.2026 09:15:10,"B6, Kenya, 2.98 km from Sotik",1848.00,28 km/h,0:00:13,0.10 km,1
4,ENA COACH - KDE 182Q,Over Revving,26.04.2026 13:16:46,"Kaplong-Narok-Maai Road, Kenya, 1.86 km from Kitet B",26.04.2026 13:16:57,"Kaplong-Narok-Maai Road, Kenya, 1.88 km from Kitet B",1856.00,46 km/h,0:00:11,0.12 km,1
5,ENA COACH - KDE 182Q,Over Revving,26.04.2026 14:03:32,"Limuru-Maimahiu Road, Kenya, 2.03 km from Escarpment",26.04.2026 14:03:43,"Limuru-Maimahiu Road, Kenya, 2.11 km from Escarpment",1836.00,46 km/h,0:00:11,0.11 km,1


In [61]:
harsh_braking_df = detail_df[detail_df["Violation"] == "Harsh Braking"].copy()
for col in ["Initial location", "Final location"]:
    harsh_braking_df[col] = harsh_braking_df[col].apply(make_location_hyperlink)
harsh_braking_df.index = range(1, len(harsh_braking_df) + 1)
HTML(harsh_braking_df.head(5).to_html(escape=False))

,Grouping,Violation,Beginning,Initial location,End,Final location,Avg. speed,Max. speed,Duration,Mileage,Count
1,ENA COACH - KDE 181Q,Harsh Braking,27.04.2026 06:18:15,"Mombasa Road, Kenya, 1.29 km from Syokimau",27.04.2026 06:18:27,"Mombasa Road, Kenya, 1.29 km from Syokimau",1.00,0 km/h,0:00:12,0.00 km,1
2,ENA COACH - KDE 181Q,Harsh Braking,27.04.2026 06:20:15,"Mombasa Road, Kenya, 1.28 km from Syokimau",27.04.2026 06:20:25,"Mombasa Road, Kenya, 1.28 km from Syokimau",1.00,0 km/h,0:00:10,0.00 km,1
3,ENA COACH - KDE 181Q,Harsh Braking,27.04.2026 06:52:08,"Mombasa Road, Kenya, 1.29 km from Syokimau",27.04.2026 06:52:30,"Mombasa Road, Kenya, 1.29 km from Syokimau",1.00,0 km/h,0:00:22,0.00 km,1
4,ENA COACH - KDE 181Q,Harsh Braking,27.04.2026 07:19:07,"Mombasa Road, Kenya, 1.31 km from Syokimau",27.04.2026 07:19:28,"Mombasa Road, Kenya, 1.31 km from Syokimau",1.00,0 km/h,0:00:21,0.00 km,1
5,ENA COACH - KDE 181Q,Harsh Braking,27.04.2026 07:38:14,"Mombasa Road, Kenya, 1.31 km from Syokimau",27.04.2026 07:38:37,"Mombasa Road, Kenya, 1.31 km from Syokimau",1.00,0 km/h,0:00:23,0.00 km,1


In [62]:
def to_number(series, keep_float=True):
    """Extract first numeric value from text like '12796 km' or '35 l/100 km'."""
    extracted = series.astype(str).str.extract(r"([0-9]+(?:\.[0-9]+)?)", expand=False)
    nums = pd.to_numeric(extracted, errors="coerce")
    if keep_float:
        return nums
    return nums.fillna(0).astype(int)


# Build a normalized summary table from available report data
summary_df = ena_coach_summary.copy()
summary_df["Distance_km"] = to_number(summary_df["Mileage in all messages"])
summary_df["Average_speed_kmh"] = to_number(summary_df["Avg. speed"])
summary_df["Avg_fuel_km_per_l"] = to_number(summary_df["Avg. mileage per unit of fuel by FLS"])
summary_df["Fuel_consumption_diesel_l"] = to_number(summary_df["Consumed by AbsFCS"])
summary_df["Fuel_drains_qty"] = to_number(summary_df["Total drains"], keep_float=False)

# Parse grouping into Drivers and Vehicle where possible
split_grouping = summary_df["Grouping"].astype(str).str.split(" - ", n=1, expand=True)
summary_df["Drivers"] = split_grouping[0].str.strip()
summary_df["Vehicle"] = split_grouping[1].fillna(split_grouping[0]).str.strip()

# Aggregations from detail table
detail_work = detail_df.copy()
detail_work["Mileage_num"] = to_number(detail_work["Mileage"])

violation_counts = (
    detail_work.groupby(["Grouping", "Violation"]).size().unstack(fill_value=0)
)

freewheel_km = (
    detail_work[detail_work["Violation"] == "Free Wheeling"]
    .groupby("Grouping")["Mileage_num"]
    .sum()
)

# Required final columns (kept in exact order requested)
required_columns = [
    "Drivers",
    "Vehicle",
    "Average fuel consumption km/l",
    "Distance  km",
    "Engine running time  hours, minutes",
    "Brake applications  #/100",
    "Harsh brake applications  #/100",
    "Harsh acceleration  #/100",
    "Idling  % of engine running time",
    "Engine overspeed  % of engine running time",
    "Powertrain coasting  % of distance",
    "Fuel consumption - Diesel  litres",
    "Fuel consumption - idling - Diesel  litres",
    "Engine running time, idling  hours, minutes",
    "Average weight  tonnes",
    "Average speed  km/h",
    "Freewheel coasting  km",
    "Brake applications  Qty",
    "Harsh brake applications  Qty",
    "# of Overspeeding Incidents",
    "# Fuel Drains",
    "# Freewheeling",
]

final_rows = []
for _, row in summary_df.iterrows():
    grouping = row["Grouping"]
    distance = row["Distance_km"]

    harsh_brake_qty = int(violation_counts.loc[grouping, "Harsh Braking"]) if (
        grouping in violation_counts.index and "Harsh Braking" in violation_counts.columns
    ) else 0

    overspeed_qty = int(violation_counts.loc[grouping, "Over Speeding"]) if (
        grouping in violation_counts.index and "Over Speeding" in violation_counts.columns
    ) else 0

    freewheel_qty = int(violation_counts.loc[grouping, "Free Wheeling"]) if (
        grouping in violation_counts.index and "Free Wheeling" in violation_counts.columns
    ) else 0

    brake_per_100 = (harsh_brake_qty / distance * 100) if pd.notna(distance) and distance > 0 else np.nan
    harsh_brake_per_100 = brake_per_100

    final_rows.append(
        {
            "Drivers": row["Drivers"],
            "Vehicle": row["Vehicle"],
            "Average fuel consumption km/l": row["Avg_fuel_km_per_l"],
            "Distance  km": distance,
            "Engine running time  hours, minutes": row.get("Engine hours", np.nan),
            "Brake applications  #/100": brake_per_100,
            "Harsh brake applications  #/100": harsh_brake_per_100,
            "Harsh acceleration  #/100": np.nan,
            "Idling  % of engine running time": np.nan,
            "Engine overspeed  % of engine running time": np.nan,
            "Powertrain coasting  % of distance": np.nan,
            "Fuel consumption - Diesel  litres": row["Fuel_consumption_diesel_l"],
            "Fuel consumption - idling - Diesel  litres": np.nan,
            "Engine running time, idling  hours, minutes": np.nan,
            "Average weight  tonnes": np.nan,
            "Average speed  km/h": row["Average_speed_kmh"],
            "Freewheel coasting  km": float(freewheel_km.get(grouping, 0.0)),
            "Brake applications  Qty": harsh_brake_qty,
            "Harsh brake applications  Qty": harsh_brake_qty,
            "# of Overspeeding Incidents": overspeed_qty,
            "# Fuel Drains": int(row["Fuel_drains_qty"]) if pd.notna(row["Fuel_drains_qty"]) else 0,
            "# Freewheeling": freewheel_qty,
        }
    )

final_driver_vehicle_table = pd.DataFrame(final_rows, columns=required_columns)

# Optional formatting for easier reading
for c in [
    "Average fuel consumption km/l",
    "Distance  km",
    "Brake applications  #/100",
    "Harsh brake applications  #/100",
    "Fuel consumption - Diesel  litres",
    "Average speed  km/h",
    "Freewheel coasting  km",
]:
    final_driver_vehicle_table[c] = pd.to_numeric(final_driver_vehicle_table[c], errors="coerce").round(2)

with pd.option_context("display.max_columns", None, "display.width", 2000, "display.max_colwidth", None):
    display(final_driver_vehicle_table)

,Drivers,Vehicle,Average fuel consumption km/l,Distance km,"Engine running time hours, minutes",Brake applications #/100,Harsh brake applications #/100,Harsh acceleration #/100,Idling % of engine running time,Engine overspeed % of engine running time,Powertrain coasting % of distance,Fuel consumption - Diesel litres,Fuel consumption - idling - Diesel litres,"Engine running time, idling hours, minutes",Average weight tonnes,Average speed km/h,Freewheel coasting km,Brake applications Qty,Harsh brake applications Qty,# of Overspeeding Incidents,# Fuel Drains,# Freewheeling
0,ENA COACH,KDE 181Q,0.0,0.15,1:33:24,3333.33,3333.33,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,0,0.0,5,5,0,0,0
1,ENA COACH,KDE 182Q,0.0,574.00,15:59:53,2.96,2.96,NaN,NaN,NaN,NaN,194.0,NaN,NaN,NaN,16,0.0,17,17,109,0,0


In [63]:
# Detailization for fuel tables (ena_coach_fillings and ena_coach_drains)
# Uses report/get_result_subrows exactly like ecodriving detailization flow.

def detailize_report_table(table_index, parent_df, col_index=0, max_rows=1000):
    all_details = []

    for i in range(len(parent_df)):
        detail_payload = {
            "svc": "report/get_result_subrows",
            "params": json.dumps(
                {
                    "tableIndex": table_index,
                    "rowIndex": i,
                    "colIndex": col_index,
                    "indexFrom": 0,
                    "indexTo": max_rows,
                }
            ),
            "sid": eid,
        }

        response = requests.post(
            "https://hst-api.wialon.com/wialon/ajax.html",
            data=detail_payload,
        )

        result = response.json()
        for row in result:
            cells = row.get("c", [])
            values = [c.get("t") if isinstance(c, dict) else c for c in cells]
            all_details.append(values)

    detail_df = pd.DataFrame(all_details)

    # If there are no subrows, keep original table as fallback
    if detail_df.empty:
        return parent_df.copy()

    # Align columns with parent dataframe headers when possible
    parent_cols = list(parent_df.columns)
    if len(detail_df.columns) >= len(parent_cols):
        detail_df = detail_df.iloc[:, :len(parent_cols)]
        detail_df.columns = parent_cols
    else:
        # If detailized rows have fewer fields, keep generic names for unmatched columns
        detail_df.columns = [f"field_{i}" for i in range(len(detail_df.columns))]

    return detail_df


# Build detailed dataframes for the two fuel tables
ena_coach_fillings = detailize_report_table(table_index=3, parent_df=ena_coach_fillings)
ena_coach_drains = detailize_report_table(table_index=4, parent_df=ena_coach_drains)

# Display detailized outputs
display(ena_coach_fillings.head())
display(ena_coach_drains.head())

,Grouping,Filling or charge time,Location,Initial fuel level,Filled,Final fuel level,Registered filling,Filling difference,Driver,Mileage
0,ENA COACH - KDE 181Q,27.04.2026 05:48:23,"Mombasa Road, Kenya, 1.28 km from Syokimau",351 l,27 l,378 l,-----,27 l,ENA Driver1,1251202 km
1,ENA COACH - KDE 182Q,-----,,-----,-----,-----,-----,0.00 l,,1276087 km


,Grouping,Initial location,Drain time,Final location,Drained,Final fuel level,Driver,Mileage,Initial fuel level
0,ENA COACH - KDE 181Q,,-----,,-----,-----,,1251202 km,-----
1,ENA COACH - KDE 182Q,,-----,,-----,-----,,1276087 km,-----
